In [17]:
import opensim as osim
import numpy as np

In [5]:
[x for x in dir(osim) if "rad" in x.lower()]

['SimTK_DEGREE_TO_RADIAN',
 'SimTK_RADIAN_TO_DEGREE',
 'TabOpConvertDegreesToRadians']

In [9]:
model = osim.Model("gait2392_simbody.osim")
state = model.initSystem()

table = osim.TimeSeriesTable("normal.mot")

# Convert degrees to radians using modern API
processor = osim.TableProcessor(table)
processor.append(osim.TabOpConvertDegreesToRadians())
table = processor.process(model)


coordSet = model.getCoordinateSet()
labels = table.getColumnLabels()

for i in range(table.getNumRows()):
    
    # Get time
    time = table.getIndependentColumn()[i]
    state.setTime(time)

    # Set coordinate values manually
    row = table.getRowAtIndex(i)

    for j in range(len(labels)):
        label = labels[j]
        if coordSet.contains(label):
            coord = coordSet.get(label)
            coord.setValue(state, row[j])

    model.realizePosition(state)

    com = model.calcMassCenterPosition(state)

    print(time, com)

0.0 ~[-0.0942051,0.994658,0.00616888]
0.02 ~[-0.0962429,0.994599,0.0057566]
0.04 ~[-0.0980441,0.994373,0.00502617]
0.06 ~[-0.0993525,0.993987,0.00391253]
0.08 ~[-0.100197,0.993497,0.00254484]
0.1 ~[-0.100591,0.992983,0.00125219]
0.12 ~[-0.100443,0.992591,0.000304618]
0.14 ~[-0.0997571,0.99232,-0.000154865]
0.16 ~[-0.0987626,0.992137,-0.00025281]
0.18 ~[-0.0976772,0.991957,-7.44305e-05]
0.2 ~[-0.0965527,0.991674,0.000164962]
0.22 ~[-0.0955061,0.991335,0.00027259]
0.24 ~[-0.094451,0.990956,0.000125273]
0.26 ~[-0.0934499,0.990642,-0.000262784]
0.28 ~[-0.0924499,0.990384,-0.000750822]
0.3 ~[-0.0915206,0.990271,-0.00132935]
0.32 ~[-0.0906359,0.990314,-0.00190214]
0.34 ~[-0.0898437,0.990552,-0.00252469]
0.36 ~[-0.0892712,0.990981,-0.00326972]
0.38 ~[-0.0889871,0.991607,-0.00413993]
0.4 ~[-0.0890231,0.992372,-0.00510799]
0.42 ~[-0.0893972,0.993202,-0.00605381]
0.44 ~[-0.0902091,0.993984,-0.0068733]
0.46 ~[-0.0913922,0.994604,-0.00746691]
0.48 ~[-0.0929013,0.995022,-0.00774769]
0.5 ~[-0.094663

### Computinh leg length (I only computed length of the upper part of the leg, check if i should change this)

In [ ]:
model = osim.Model("gait2392_simbody.osim")
state = model.initSystem()

# Get hip and knee joints
hip_joint = model.getJointSet().get("hip_r")
knee_joint = model.getJointSet().get("knee_r")

# Get their locations in ground
hip_location = hip_joint.getParentFrame().getPositionInGround(state)
knee_location = knee_joint.getChildFrame().getPositionInGround(state)

thigh_length = np.linalg.norm([
    hip_location.get(i) - knee_location.get(i)
    for i in range(3)
])

print("Right thigh length (m):", thigh_length)

Right thigh length (m): 0.395846273754769


In [19]:
# check if it matches right thigh length in model
hip_l = model.getJointSet().get("hip_l")
knee_l = model.getJointSet().get("knee_l")

hip_loc_l = hip_l.getParentFrame().getPositionInGround(state)
knee_loc_l = knee_l.getChildFrame().getPositionInGround(state)

thigh_length_l = np.linalg.norm([
    hip_loc_l.get(i) - knee_loc_l.get(i)
    for i in range(3)
])

print("Left thigh length (m):", thigh_length_l)

Left thigh length (m): 0.395846273754769


### Computing a gait cycle, since right heel touches ground until it touches ground again

In [24]:
times = []
knee_r_values = []

coordSet = model.getCoordinateSet()
knee_coord = coordSet.get("knee_angle_r")

for i in range(table.getNumRows()):
    time = table.getIndependentColumn()[i]
    state.setTime(time)

    row = table.getRowAtIndex(i)
    labels = table.getColumnLabels()

    for j in range(len(labels)):
        label = labels[j]
        if coordSet.contains(label):
            coord = coordSet.get(label)
            coord.setValue(state, row[j])

    model.realizePosition(state)

    times.append(time)
    knee_r_values.append(knee_coord.getValue(state))

    knee_array = np.array(knee_r_values)
times_array = np.array(times)

# Find local minima (Heel strike approximately corresponds to knee extension (minimum flexion).)
min_indices = []

for i in range(1, len(knee_array)-1):
    if knee_array[i] < knee_array[i-1] and knee_array[i] < knee_array[i+1]:
        min_indices.append(i)

print("Detected minima at:", min_indices[:3])

start_idx = min_indices[0]
end_idx = min_indices[1]

cycle_times = times_array[start_idx:end_idx]

# normalize time
cycle_duration = cycle_times[-1] - cycle_times[0]
normalized_time = (cycle_times - cycle_times[0]) / cycle_duration * 100

Detected minima at: [6, 36]


Then: thinking how to diversify data. check how to do an efficient loop for producing thousands of training instances (one gait cycle per model). mathis will have the information on what exactly i should vary, but i should be able to alaready have this in my code.
check in the lecture notes exactly how we will use sbi, to give the training data in an appropiate way for this.

We need to discuss what type of data we will recieve from the sensors. I think with gps we could have global coordinates, so it is okay to assume that we have this.
### This block: Extracts one gait cycle, Computes global foot positions, Centers them, Resamples to 100 points, Computes leg length

In [1]:
import opensim as osim
import numpy as np
from scipy.interpolate import interp1d

In [2]:
def resample(normalized_time, signal):
    target_time = np.linspace(0, 100, 100)
    return interp1d(normalized_time, signal)(target_time)

In [11]:
def gait_cycle(table, model, coordSet, state, labels):
    # gait cycle (minimal knee angle)
    times = []
    knee_values = []

    knee_coord = coordSet.get("knee_angle_r")

    for i in range(table.getNumRows()):
        time = table.getIndependentColumn()[i]
        state.setTime(time)

        row = table.getRowAtIndex(i)

        for j in range(len(labels)):
            label = labels[j]
            if coordSet.contains(label):
                coordSet.get(label).setValue(state, row[j])

        model.realizePosition(state)

        times.append(time)
        knee_values.append(knee_coord.getValue(state))

    times = np.array(times)
    knee_values = np.array(knee_values)

    # Find minima
    min_indices = []
    for i in range(1, len(knee_values)-1):
        if knee_values[i] < knee_values[i-1] and knee_values[i] < knee_values[i+1]:
            min_indices.append(i)

    start_idx = min_indices[0]
    end_idx = min_indices[1]

    # Normalize time 0–100
    cycle_times = times[start_idx:end_idx]
    cycle_duration = cycle_times[-1] - cycle_times[0]
    normalized_time = (cycle_times - cycle_times[0]) / cycle_duration * 100

    return start_idx, end_idx, normalized_time

In [24]:
def foot_coordinates_cycle(model, state, table, start_idx, end_idx, normalized_time, labels, coordSet):
    foot_r = model.getBodySet().get("calcn_r")
    foot_l = model.getBodySet().get("calcn_l")

    foot_rx, foot_ry, foot_rz = [], [], []
    foot_lx, foot_ly, foot_lz = [], [], []

    for i in range(start_idx, end_idx):

        time = table.getIndependentColumn()[i]
        state.setTime(time)

        row = table.getRowAtIndex(i)

        for j in range(len(labels)):
            label = labels[j]
            if coordSet.contains(label):
                coordSet.get(label).setValue(state, row[j])

        model.realizePosition(state)

        pos_r = foot_r.getPositionInGround(state)
        pos_l = foot_l.getPositionInGround(state)

        foot_rx.append(pos_r.get(0))
        foot_ry.append(pos_r.get(1))
        foot_rz.append(pos_r.get(2))

        foot_lx.append(pos_l.get(0))
        foot_ly.append(pos_l.get(1))
        foot_lz.append(pos_l.get(2))

    foot_rx = np.array(foot_rx)
    foot_ry = np.array(foot_ry)
    foot_rz = np.array(foot_rz)

    foot_lx = np.array(foot_lx)
    foot_ly = np.array(foot_ly)
    foot_lz = np.array(foot_lz)


    # Center per gait cycle (remove drift)
    foot_rx -= foot_rx[0]
    foot_ry -= foot_ry[0]
    foot_rz -= foot_rz[0]

    foot_lx -= foot_lx[0]
    foot_ly -= foot_ly[0]
    foot_lz -= foot_lz[0]

    
    foot_rx = resample(normalized_time, foot_rx)
    foot_ry = resample(normalized_time, foot_ry)
    foot_rz = resample(normalized_time, foot_rz)

    foot_lx = resample(normalized_time, foot_lx)
    foot_ly = resample(normalized_time, foot_ly)
    foot_lz = resample(normalized_time, foot_lz)
    return foot_rx, foot_ry, foot_rz, foot_lx, foot_ly, foot_lz

In [17]:
def compute_leg_length(model, state):
    hip_r = model.getJointSet().get("hip_r")
    ankle_r = model.getJointSet().get("ankle_r")

    hip_pos = hip_r.getParentFrame().getPositionInGround(state)
    ankle_pos = ankle_r.getChildFrame().getPositionInGround(state)

    leg_length = np.linalg.norm([
        hip_pos.get(i) - ankle_pos.get(i)
        for i in range(3)
    ])
    return leg_length

In [23]:
def generate_sample():
    model = osim.Model("gait2392_simbody.osim")
    # modify model here to add variability

    state = model.initSystem()

    table = osim.TimeSeriesTable("normal.mot")

    # convert table from degrees to radians
    processor = osim.TableProcessor(table)
    processor.append(osim.TabOpConvertDegreesToRadians())
    table = processor.process(model)

    coordSet = model.getCoordinateSet()
    labels = table.getColumnLabels()

    # gait cycle (minimal knee angle)
    start_idx, end_idx, normalized_time = gait_cycle(table, model, coordSet, state, labels)

    leg_length = compute_leg_length(model, state)

    # Extract foot positions over cycle
    foot_rx, foot_ry, foot_rz, foot_lx, foot_ly, foot_lz = foot_coordinates_cycle(model, state, table, start_idx, end_idx, normalized_time, labels, coordSet)

    # shape: (100, 6)
    foot_data = np.stack([
        foot_rx, foot_ry, foot_rz,
        foot_lx, foot_ly, foot_lz
    ], axis=1)

    return leg_length, foot_data

In [25]:
print("Sample leg length and foot data:")
leg_length, foot_data = generate_sample()
print("Leg length:", leg_length)
print("Foot data shape:", foot_data.shape)

Sample leg length and foot data:
Leg length: 0.825744999552779
Foot data shape: (100, 6)
